# Manchester Heat-Aware Routing Analysis

Analyse the impact of heat-aware routing on pedestrian and cyclist travel.

### Scenarios:
- **peak:** Hottest hour (17:00, 30 Sep 2025, WBGT up to 26.26°C)
- **commute:** Monday 08:00 (typical weekday morning)
- **weekend:** Saturday 12:00 (weekend midday)

### Two-Case Comparison per Scenario:
1. **Fastest (w=1.0):** Time-minimising routes
2. **Heat-optimal:** Hypervolume-selected trade-off (10-weight sweep)

### Data Source:
- URBCLIM hourly WBGT at 200m for Greater Manchester
- MATSim static routing with Pareto weight sweep (10 weights: 0.8^{0,2,4,...,18})

In [ ]:
# @title Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os, pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')

routing_data_path = '/content/drive/MyDrive/Colab Notebooks/phd_data/routing/'

In [ ]:
# @title Analyse heat routing results for all scenarios — LaTeX-ready
def parse_list(s):
    s = s.strip()
    if s.startswith('[') and s.endswith(']'):
        s = s[1:-1]
    if not s.strip():
        return []
    return [x.strip() for x in s.split(',')]

def fmt_median_mean(vals):
    return f'{np.median(vals):.2f}', f'{np.mean(vals):.2f}'

scenarios = ['peak', 'commute', 'weekend']
scenario_labels = {'peak': 'Peak', 'commute': 'Commute', 'weekend': 'Weekend'}
mode_labels = {'walk': 'Walk', 'bike': 'Cycle'}

# Accumulate data for LaTeX tables
latex_travel = []
latex_exposure = []
latex_jaccard = []

for scenario in scenarios:
    dirs = sorted([d for d in os.listdir(routing_data_path)
                   if d.startswith(f'results_heat_{scenario}_')])
    latest = routing_data_path + dirs[-1] + '/' if dirs else routing_data_path
    print(f'\n{"="*60}')
    print(f'  Scenario: {scenario}')
    print(f'  Loading from: {latest}')

    for mode in ["walk", "bike"]:
        fast_file = latest + f"routes_fastest_{mode}.csv"
        opt_file = latest + f"routes_heat_{mode}.csv"
        if not os.path.exists(fast_file) or not os.path.exists(opt_file):
            continue

        fastest = pd.read_csv(fast_file)
        optimal = pd.read_csv(opt_file)

        for col in ["path", "links"]:
            fastest[col] = fastest[col].apply(parse_list)
            optimal[col] = optimal[col].apply(parse_list)

        merged = fastest[["person","travel_time","heat_exposure"]].rename(
            columns={"travel_time": "time_c1", "heat_exposure": "heat_c1"}).merge(
            optimal[["person","travel_time","heat_exposure","weight"]].rename(
                columns={"travel_time": "time_c2", "heat_exposure": "heat_c2"}),
            on="person")

        # --- Travel time stats ---
        t1 = merged["time_c1"] / 60.0
        t2 = merged["time_c2"] / 60.0
        t1_med, t1_mean = fmt_median_mean(t1)
        t2_med, t2_mean = fmt_median_mean(t2)
        sl = scenario_labels[scenario]
        ml = mode_labels[mode]
        latex_travel.append(f'{ml} & {sl} & Median & {t1_med} & {t2_med} \\\\')
        latex_travel.append(f'       &       & Mean   & {t1_mean} & {t2_mean} \\\\')

        # --- WBGT exposure stats ---
        h1_mean = merged["heat_c1"] / (merged["time_c1"] / 60.0)  # mean WBGT
        h2_mean = merged["heat_c2"] / (merged["time_c2"] / 60.0)
        h1_cum = merged["heat_c1"]  # cumulative
        h2_cum = merged["heat_c2"]
        h1_mean_v, h1_cum_v = fmt_median_mean(h1_mean)
        h2_mean_v, h2_cum_v = fmt_median_mean(h2_mean)
        h1_cum_med, h1_cum_mean = fmt_median_mean(h1_cum)
        h2_cum_med, h2_cum_mean = fmt_median_mean(h2_cum)
        latex_exposure.append(f'{ml} & {sl} & Mean WBGT & {h1_mean_v} & {h2_mean_v} \\\\')
        latex_exposure.append(f'       &       & Cumul. WBGT & {h1_cum_med} & {h2_cum_med} \\\\')

        # --- Jaccard ---
        def jaccard(a, b):
            inter = len(set(a) & set(b))
            union = len(set(a) | set(b))
            return inter / union if union else 0

        merged["jaccard"] = [jaccard(
            fastest[fastest["person"]==p].iloc[0]["path"],
            optimal[optimal["person"]==p].iloc[0]["path"])
            for p in merged["person"]]
        j_mean, j_med = fmt_median_mean(merged["jaccard"])
        latex_jaccard.append(f'{ml} & {sl} & {j_mean} & {j_med} \\\\')

print('\n\n=== LATEX: TRAVEL TIME (tab:heat_travel_times) ===')
for line in latex_travel:
    print(line)

print('\n\n=== LATEX: WBGT EXPOSURE (tab:heat_exposure) ===')
for line in latex_exposure:
    print(line)

print('\n\n=== JACCARD (for text) ===')
for line in latex_jaccard:
    print(line)

In [ ]:
# @title Visualise heat routing results
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['hatch.linewidth'] = 0.5

# Rebuild data structures for plotting
plot_data = []  # list of dicts
for scenario in scenarios:
    dirs = sorted([d for d in os.listdir(routing_data_path)
                   if d.startswith(f'results_heat_{scenario}_')])
    latest = routing_data_path + dirs[-1] + '/' if dirs else routing_data_path
    for mode in ['walk', 'bike']:
        fast_file = latest + f'routes_fastest_{mode}.csv'
        opt_file = latest + f'routes_heat_{mode}.csv'
        if not os.path.exists(fast_file) or not os.path.exists(opt_file):
            continue
        fastest = pd.read_csv(fast_file)
        optimal = pd.read_csv(opt_file)
        for col in ['path', 'links']:
            fastest[col] = fastest[col].apply(parse_list)
            optimal[col] = optimal[col].apply(parse_list)
        merged = fastest[['person','travel_time','heat_exposure']].rename(
            columns={'travel_time': 'time_c1', 'heat_exposure': 'heat_c1'}).merge(
            optimal[['person','travel_time','heat_exposure','weight']].rename(
                columns={'travel_time': 'time_c2', 'heat_exposure': 'heat_c2'}),
            on='person')
        merged['time_c1_min'] = merged['time_c1'] / 60
        merged['time_c2_min'] = merged['time_c2'] / 60
        merged['heat_c1_mean'] = merged['heat_c1'] / (merged['time_c1'] / 60)
        merged['heat_c2_mean'] = merged['heat_c2'] / (merged['time_c2'] / 60)
        def jaccard(a, b):
            inter = len(set(a) & set(b))
            union = len(set(a) | set(b))
            return inter / union if union else 0
        merged['jaccard'] = [jaccard(
            fastest[fastest['person']==p].iloc[0]['path'],
            optimal[optimal['person']==p].iloc[0]['path']) for p in merged['person']]
        plot_data.append({'scenario': scenario, 'mode': mode, 'df': merged})

# --- Figure 1: Travel time ---
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3), sharey=True)
for idx, mode in enumerate(['walk', 'bike']):
    ax = axes[idx]
    scens = []
    c1_meds, c1_means = [], []
    c2_meds, c2_means = [], []
    for s in scenarios:
        d = [p for p in plot_data if p['scenario']==s and p['mode']==mode]
        if not d: continue
        df = d[0]['df']
        scens.append(scenario_labels[s])
        c1_meds.append(df['time_c1_min'].median())
        c1_means.append(df['time_c1_min'].mean())
        c2_meds.append(df['time_c2_min'].median())
        c2_means.append(df['time_c2_min'].mean())
    x = np.arange(len(scens))
    w = 0.3
    ax.bar(x - w/2, c1_meds, w, label='Fastest', color='#4C72B0')
    ax.bar(x + w/2, c2_meds, w, label='Heat-aware', color='#DD8452')
    ax.set_xticks(x)
    ax.set_xticklabels(scens, fontsize=8)
    ax.set_title(f'{"Walk" if mode=="walk" else "Cycle"}', fontsize=10)
    ax.set_ylabel('Median travel time (min)', fontsize=8)
    ax.tick_params(labelsize=7)
    if idx == 1:
        ax.legend(fontsize=7)
fig.suptitle('Travel Time: Fastest vs Heat-Aware', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{routing_data_path}heat_travel_time.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Figure 2: WBGT exposure ---
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3), sharey=False)
for idx, mode in enumerate(['walk', 'bike']):
    ax = axes[idx]
    scens = []
    h1_meds, h2_meds = [], []
    for s in scenarios:
        d = [p for p in plot_data if p['scenario']==s and p['mode']==mode]
        if not d: continue
        df = d[0]['df']
        scens.append(scenario_labels[s])
        h1_meds.append(df['heat_c1_mean'].median())
        h2_meds.append(df['heat_c2_mean'].median())
    x = np.arange(len(scens))
    w = 0.3
    ax.bar(x - w/2, h1_meds, w, label='Fastest', color='#4C72B0')
    ax.bar(x + w/2, h2_meds, w, label='Heat-aware', color='#DD8452')
    ax.set_xticks(x)
    ax.set_xticklabels(scens, fontsize=8)
    ax.set_title(f'{"Walk" if mode=="walk" else "Cycle"}', fontsize=10)
    ax.set_ylabel('Median mean WBGT (\u00b0C)', fontsize=8)
    ax.tick_params(labelsize=7)
    if idx == 1:
        ax.legend(fontsize=7)
fig.suptitle('Heat Exposure: Fastest vs Heat-Aware', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{routing_data_path}heat_exposure.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Figure 3: Jaccard similarity ---
fig, ax = plt.subplots(figsize=(4, 3))
modes_plot = []
scens_plot = []
j_means = []
for mode in ['walk', 'bike']:
    for s in scenarios:
        d = [p for p in plot_data if p['scenario']==s and p['mode']==mode]
        if not d: continue
        modes_plot.append('Walk' if mode=='walk' else 'Cycle')
        scens_plot.append(scenario_labels[s])
        j_means.append(d[0]['df']['jaccard'].mean())
x = np.arange(len(scens_plot))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860']
bars = ax.bar(x, j_means, color=colors[:len(x)], width=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f'{m}\n{s}' for m, s in zip(modes_plot, scens_plot)], fontsize=7)
ax.set_ylabel('Mean Jaccard similarity', fontsize=9)
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=0.5)
ax.tick_params(labelsize=7)
fig.suptitle('Route Overlap (Jaccard)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{routing_data_path}heat_jaccard.png', dpi=300, bbox_inches='tight')
plt.show()

print('Figures saved to:', routing_data_path)